In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams.update({'errorbar.capsize': 3})
plt.rcParams['xtick.labelsize']= 13
plt.rcParams['ytick.labelsize']= 13
plt.rcParams.update({'axes.labelsize': 16})

In [24]:
from astropy.io import ascii

In [43]:
input_table_loc="expres_2024q4.csv"
input_year=2024
windows_file_loc="window_overlaps_"+str(input_year)+".csv"

orig_download_script="expres_download_2024q4_orig.sh"


In [44]:
input_table = ascii.read(input_table_loc)

windows_table = ascii.read(windows_file_loc)


In [45]:
input_table.colnames

['filename',
 'instrument',
 'obsdate',
 'exptime',
 'obsmode',
 'obstype',
 'program',
 'sunagl',
 'seeing',
 'ccfjdsum',
 'ccfrvmod',
 'dvrms',
 'swversion',
 'extsnr',
 'flagged']

In [46]:
#for each line in the input table, get the value from 'ccfjdsum'
good_files = []
for row in input_table:
    inwin = False
    ccfjdsum_value = row['ccfjdsum']
    #check is snr>200:
    if row['extsnr'] >= 200:
        

        #now check to see if it is in the windows listed in the window table 
        # (i.e. bigger than the start_jd column and smaller than the end_jd column)
        for window_row in windows_table:
            if ccfjdsum_value > window_row['start_jd'] and ccfjdsum_value < window_row['end_jd']:
                inwin = True
                #add the file name to the list of good files
                good_files.append(row['filename'])


In [47]:
len(good_files)

0

In [48]:
#read in the original download script
with open(orig_download_script, 'r') as f:
    orig_script = f.readlines()

In [49]:
new_script=[]
for item in orig_script:
    #if the line doesn't start with "wget --load-cookies",
    #  add the line to the new script
    if not item.startswith("wget --load-cookies"):
        new_script.append(item)
    else:
        filename=item.split()[4]
        if filename in good_files:
            new_script.append(item)


In [50]:
len(new_script)

17

In [51]:
#write the new script to a file
#new file name should substitue _orig 
new_file_name = orig_download_script.replace("_orig", "")
with open(new_file_name, 'w') as f:
    for item in new_script:
        f.write(item) 